# PySR / ML BSM Exclusion Workflow Report

Supervisor-facing draft report.

**Status:** all model results, interpretations, dataset conventions, and supervisor-facing claims in this notebook are provisional, unverified, and pending review.

## Directed Message To Supervisor

Dear Professor,

This notebook summarizes the current reproducible workflow for the BSM exclusion surrogate-model study. The main update is that an approved engineered-mass machine-learning candidate has reproduced AUC >= 0.97 on the fixed seed-42 stratified split. However, robustness across repeated splits and cross-validation is not yet established. The final PySR symbolic-search step has not yet been launched for this candidate.

I am using this notebook to separate what has been observed from what still requires scientific review.

## Research Objective

The objective is to develop a reviewed and reproducible comparison between PySR symbolic-regression models and standard machine-learning surrogate models for binary BSM exclusion classification.

This notebook is a draft research report and review artifact. It does not establish final thesis results, accepted physics interpretations, or accepted dataset conventions.

In [1]:
from pathlib import Path
import json

import pandas as pd
import yaml
from IPython.display import Markdown, display

STATUS = "provisional, unverified, pending review"

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks" and (ROOT.parent / "configs").exists():
    ROOT = ROOT.parent
elif not (ROOT / "configs").exists() and (ROOT.parent / "configs").exists():
    ROOT = ROOT.parent

def repo_path(relative_path: str) -> Path:
    return ROOT / relative_path

def load_yaml(relative_path: str):
    with repo_path(relative_path).open("r", encoding="utf-8") as handle:
        return yaml.safe_load(handle)

def load_json_if_present(relative_path: str):
    path = repo_path(relative_path)
    if not path.exists():
        return None
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)

def read_text_if_present(relative_path: str) -> str | None:
    path = repo_path(relative_path)
    if not path.exists():
        return None
    return path.read_text(encoding="utf-8")

def display_markdown_file(relative_path: str, title: str):
    text = read_text_if_present(relative_path)
    if text is None:
        display(Markdown(f"**Missing generated summary:** `{relative_path}`. This section remains pending generated evidence."))
    else:
        display(Markdown(text))

dataset_config = load_yaml("configs/datasets/masses_exclusions.yaml")
candidate_config = load_yaml("configs/runs/masses_exclusions_auc97_candidate.yaml")
candidate_metadata = load_json_if_present("outputs/runs/masses_exclusions_auc97_candidate/auc97_candidate_metadata.json")

candidate_metrics_path = repo_path("outputs/runs/masses_exclusions_auc97_candidate/auc97_candidate_metrics.csv")
candidate_metrics = pd.read_csv(candidate_metrics_path) if candidate_metrics_path.exists() else pd.DataFrame()

display(Markdown(f"Loaded report inputs from `{ROOT}`. Notebook result status: **{STATUS}**."))

Loaded report inputs from `/Users/vaheedgorgeen/workspace/train-pysr`. Notebook result status: **provisional, unverified, pending review**.

## Dataset Summary

The active registered dataset is `masses_exclusions`, stored at `data/raw/masses_exclusions.csv`. The raw data must remain unchanged. The dataset registry records 12,280 rows and 4 columns: `mchi1`, `mchipm1`, `Final_CLs`, and `exclusion`.

`exclusion` is the binary target. The observed class counts are 0: 2,263 and 1: 10,017. Units for the mass columns, detailed target-label semantics, preprocessing rules, split protocol, and final class-imbalance policy remain review items unless declared in a reviewed run config.

In [2]:
columns = dataset_config["schema"]["columns"]
rows = []
for column_name, info in columns.items():
    summary = info.get("observed_summary", {})
    rows.append({
        "column": column_name,
        "role": info.get("role"),
        "dtype": info.get("observed_dtype"),
        "units": info.get("units"),
        "missing_values": info.get("missing_values"),
        "min": summary.get("min"),
        "max": summary.get("max"),
        "mean": summary.get("mean"),
        "std": summary.get("std"),
    })

display(pd.DataFrame(rows))
display(pd.DataFrame([
    {"label": label, "count": count}
    for label, count in dataset_config["target"]["observed_class_counts"].items()
]))

,column,role,dtype,units,missing_values,min,max,mean,std
0,mchi1,feature,float64,requires_review,0,0.222243,1936.32,506.900000,369.587000
1,mchipm1,feature,float64,requires_review,0,92.042700,1981.43,708.531000,466.948000
2,Final_CLs,audit_only,float64,requires_review,0,0.000000,1.00,0.608857,0.398223
3,exclusion,target,int64,not_applicable,0,NaN,NaN,NaN,NaN


,label,count
0,0,2263
1,1,10017


### Dataset Audit Output

The following generated audit summary is displayed as supporting evidence. It remains provisional, unverified, and pending review.

In [3]:
display_markdown_file(
    "outputs/runs/masses_exclusions_audit/dataset_audit.md",
    "Dataset Audit",
)

# Dataset Audit

Status: provisional, unverified, pending review.

## Provenance

- Timestamp UTC: `2026-06-03T23:12:14.107435+00:00`
- Command: `scripts/audit_dataset.py --config configs/runs/masses_exclusions_audit.yaml`
- Config path: `configs/runs/masses_exclusions_audit.yaml`
- Dataset path: `data/raw/masses_exclusions.csv`
- Run id: `masses_exclusions_audit`
- Dataset id: `masses_exclusions`

## Summary

- Shape: 12280 rows, 4 columns
- Columns: `mchi1`, `mchipm1`, `Final_CLs`, `exclusion`
- Duplicate rows: 0

## Dtypes

- `mchi1`: `float64`
- `mchipm1`: `float64`
- `Final_CLs`: `float64`
- `exclusion`: `int64`

## Missing Values

- `mchi1`: 0
- `mchipm1`: 0
- `Final_CLs`: 0
- `exclusion`: 0

## Target

- Target column: `exclusion`
- Unique values: ['0', '1']
- Class counts: {'0': 2263, '1': 10017}

## Numeric Summary

- `mchi1`: {'count': 12280, 'min': 0.222243372, 'max': 1936.32098, 'mean': 506.8997794015612, 'std': 369.5868528519316}
- `mchipm1`: {'count': 12280, 'min': 92.0426893, 'max': 1981.43446, 'mean': 708.5308036215229, 'std': 466.9481141042298}
- `Final_CLs`: {'count': 12280, 'min': 0.0, 'max': 1.0, 'mean': 0.6088565005203781, 'std': 0.3982225478960791}
- `exclusion`: {'count': 12280, 'min': 0.0, 'max': 1.0, 'mean': 0.8157166123778502, 'std': 0.38773091565402823}

## Mass Checks

- Negative mass counts: {'mchi1': 0, 'mchipm1': 0}
- Mass ordering check: {'checked': True, 'mchipm1_greater_than_or_equal_mchi1_always': True, 'violations': 0}

## Audit-Only Columns

- Audit-only columns: ['Final_CLs']
- Final_CLs/target correlation: {'columns': ['Final_CLs', 'exclusion'], 'pearson': 0.7181377323447221, 'interpretation': 'Numerical association only; no physics meaning or exclusion rule is inferred.'}

## Notes

- Raw data was read only and not modified.
- Final_CLs is treated as audit-only.
- No exclusion rule is inferred.
- No empirical model performance is claimed.


## Feature Conventions

The default approved-feature starting point is the two mass columns, `mchi1` and `mchipm1`. The current AUC candidate uses the reviewed run configuration `mass_engineered_v1`, which augments the two base masses with mass-gap, mass-sum, mass-ratio, and logarithmic engineered features.

These engineered features are useful for the current candidate workflow, but their scientific acceptability remains review-sensitive. `Final_CLs` is audit-only and diagnostic; it is not used as an approved feature in this candidate notebook.

In [4]:
feature_set = candidate_config["feature_set"]
feature_rows = []
for column in feature_set["base_columns"]:
    feature_rows.append({"kind": "base", "feature_or_expression": column})
for expression in feature_set["derived_features"]:
    feature_rows.append({"kind": "derived", "feature_or_expression": expression})

display(pd.DataFrame(feature_rows))

,kind,feature_or_expression
0,base,mchi1
1,base,mchipm1
2,derived,delta_m = mchipm1 - mchi1
3,derived,sum_m = mchipm1 + mchi1
4,derived,ratio_m = mchipm1 / mchi1
5,derived,log_mchi1 = log1p(mchi1)
6,derived,log_mchipm1 = log1p(mchipm1)
7,derived,"log_delta_m = log1p(max(delta_m, 0))"


## Evaluation Protocol

ROC/AUC must be computed from continuous model scores, not hard class labels. Threshold-dependent quantities such as accuracy or confusion matrices are separate from the AUC criterion and are not used here as evidence for the AUC target.

The current candidate reproduction uses a stratified fixed split with `test_size = 0.2` and `random_seed = 42`. Class imbalance is handled explicitly with `compute_sample_weight`, `class_weight = balanced`, applied to the training split only.

In [5]:
protocol_rows = [
    {"field": "run_id", "value": candidate_config["run_id"]},
    {"field": "dataset_id", "value": candidate_config["dataset_id"]},
    {"field": "target", "value": candidate_config["target"]},
    {"field": "positive_label", "value": candidate_config["positive_label"]},
    {"field": "auc_rule", "value": candidate_config["auc_rule"]},
    {"field": "model_family", "value": candidate_config["model"]["family"]},
    {"field": "model_variant", "value": candidate_config["model"]["variant"]},
    {"field": "candidate_split", "value": candidate_config["candidate_split"]},
    {"field": "class_imbalance_strategy", "value": candidate_config["class_imbalance_strategy"]},
    {"field": "review_status", "value": candidate_config["review_status"]},
]
display(pd.DataFrame(protocol_rows))

,field,value
0,run_id,masses_exclusions_auc97_candidate
1,dataset_id,masses_exclusions
2,target,exclusion
3,positive_label,requires_review
4,auc_rule,continuous_scores_only
5,model_family,histogram_gradient_boosting
6,model_variant,hist_gradient_boosting_grid_01
7,candidate_split,"{'split_method': 'stratified', 'test_size': 0...."
8,class_imbalance_strategy,"{'method': 'compute_sample_weight', 'class_wei..."
9,review_status,draft


## Baseline And Strong-Baseline Summary

The strong-baseline run reports a best approved feature-set ROC-AUC of 0.975742 for `mass_engineered_v1` with `hist_gradient_boosting_grid_01`. This is an observed run output only and remains provisional, unverified, and pending review.

The same summary also reports diagnostic `Final_CLs` feature sets with AUC values near or equal to 1.0. Those entries are diagnostic checks only and are not accepted thesis evidence.

In [6]:
display_markdown_file(
    "outputs/runs/masses_exclusions_strong_baselines_v1/strong_summary.md",
    "Strong Baseline Summary",
)

# Strong Baseline Summary

Status: provisional, unverified, pending review.

This summary reports observed run outputs only. It does not establish accepted thesis results.

## Best Observed AUC

- Best approved feature-set ROC-AUC: 0.975742 (mass_engineered_v1, hist_gradient_boosting_grid_01).
- Best diagnostic Final_CLs ROC-AUC: 1.000000 (final_cls_only_diagnostic, tuned_random_forest_01).
- AUC > 0.97 observed in this run; pending review.

## Ranked Results

| Rank | Feature set | Type | Model variant | ROC-AUC | Average precision | Note |
| ---: | --- | --- | --- | ---: | ---: | --- |
| 1 | `final_cls_only_diagnostic` | `diagnostic` | `tuned_random_forest_01` | 1.000000 | 1.000000 | diagnostic only |
| 2 | `final_cls_only_diagnostic` | `diagnostic` | `tuned_random_forest_02` | 1.000000 | 1.000000 | diagnostic only |
| 3 | `final_cls_only_diagnostic` | `diagnostic` | `tuned_random_forest_03` | 1.000000 | 1.000000 | diagnostic only |
| 4 | `final_cls_only_diagnostic` | `diagnostic` | `tuned_random_forest_04` | 1.000000 | 1.000000 | diagnostic only |
| 5 | `final_cls_only_diagnostic` | `diagnostic` | `extra_trees_01` | 1.000000 | 1.000000 | diagnostic only |
| 6 | `final_cls_only_diagnostic` | `diagnostic` | `extra_trees_02` | 1.000000 | 1.000000 | diagnostic only |
| 7 | `final_cls_only_diagnostic` | `diagnostic` | `extra_trees_03` | 1.000000 | 1.000000 | diagnostic only |
| 8 | `final_cls_only_diagnostic` | `diagnostic` | `extra_trees_04` | 1.000000 | 1.000000 | diagnostic only |
| 9 | `final_cls_only_diagnostic` | `diagnostic` | `hist_gradient_boosting_grid_01` | 1.000000 | 1.000000 | diagnostic only |
| 10 | `final_cls_only_diagnostic` | `diagnostic` | `hist_gradient_boosting_grid_02` | 1.000000 | 1.000000 | diagnostic only |
| 11 | `final_cls_only_diagnostic` | `diagnostic` | `hist_gradient_boosting_grid_03` | 1.000000 | 1.000000 | diagnostic only |
| 12 | `final_cls_only_diagnostic` | `diagnostic` | `hist_gradient_boosting_grid_04` | 1.000000 | 1.000000 | diagnostic only |
| 13 | `final_cls_only_diagnostic` | `diagnostic` | `logistic_polynomial_degree_3_01` | 1.000000 | 1.000000 | diagnostic only |
| 14 | `final_cls_only_diagnostic` | `diagnostic` | `logistic_polynomial_degree_3_02` | 1.000000 | 1.000000 | diagnostic only |
| 15 | `final_cls_only_diagnostic` | `diagnostic` | `logistic_polynomial_degree_3_03` | 1.000000 | 1.000000 | diagnostic only |
| 16 | `final_cls_only_diagnostic` | `diagnostic` | `rbf_svm_grid_01` | 1.000000 | 1.000000 | diagnostic only |
| 17 | `final_cls_only_diagnostic` | `diagnostic` | `rbf_svm_grid_02` | 1.000000 | 1.000000 | diagnostic only |
| 18 | `final_cls_only_diagnostic` | `diagnostic` | `rbf_svm_grid_03` | 1.000000 | 1.000000 | diagnostic only |
| 19 | `final_cls_only_diagnostic` | `diagnostic` | `rbf_svm_grid_04` | 1.000000 | 1.000000 | diagnostic only |
| 20 | `final_cls_only_diagnostic` | `diagnostic` | `knn_grid_01` | 1.000000 | 1.000000 | diagnostic only |
| 21 | `final_cls_only_diagnostic` | `diagnostic` | `knn_grid_02` | 1.000000 | 1.000000 | diagnostic only |
| 22 | `final_cls_only_diagnostic` | `diagnostic` | `knn_grid_03` | 1.000000 | 1.000000 | diagnostic only |
| 23 | `final_cls_only_diagnostic` | `diagnostic` | `knn_grid_04` | 1.000000 | 1.000000 | diagnostic only |
| 24 | `masses_plus_final_cls_diagnostic` | `diagnostic` | `tuned_random_forest_01` | 1.000000 | 1.000000 | diagnostic only |
| 25 | `masses_plus_final_cls_diagnostic` | `diagnostic` | `tuned_random_forest_02` | 1.000000 | 1.000000 | diagnostic only |
| 26 | `masses_plus_final_cls_diagnostic` | `diagnostic` | `tuned_random_forest_03` | 1.000000 | 1.000000 | diagnostic only |
| 27 | `masses_plus_final_cls_diagnostic` | `diagnostic` | `tuned_random_forest_04` | 1.000000 | 1.000000 | diagnostic only |
| 28 | `masses_plus_final_cls_diagnostic` | `diagnostic` | `hist_gradient_boosting_grid_01` | 1.000000 | 1.000000 | diagnostic only |
| 29 | `masses_plus_final_cls_diagnostic` | `diagnostic` | `hist_gradient_boosting_grid_02` | 1.000000 | 1.000000 | diagnostic only |
| 30 | `masses_plus_final_cls_diagnostic` | `diagnostic` | `hist_gradient_boosting_grid_03` | 1.000000 | 1.000000 | diagnostic only |
| 31 | `masses_plus_final_cls_diagnostic` | `diagnostic` | `hist_gradient_boosting_grid_04` | 1.000000 | 1.000000 | diagnostic only |
| 32 | `masses_plus_final_cls_diagnostic` | `diagnostic` | `rbf_svm_grid_03` | 0.999996 | 0.999999 | diagnostic only |
| 33 | `masses_plus_final_cls_diagnostic` | `diagnostic` | `extra_trees_04` | 0.999989 | 0.999998 | diagnostic only |
| 34 | `masses_plus_final_cls_diagnostic` | `diagnostic` | `extra_trees_02` | 0.999989 | 0.999998 | diagnostic only |
| 35 | `masses_plus_final_cls_diagnostic` | `diagnostic` | `rbf_svm_grid_04` | 0.999985 | 0.999997 | diagnostic only |
| 36 | `masses_plus_final_cls_diagnostic` | `diagnostic` | `extra_trees_01` | 0.999983 | 0.999996 | diagnostic only |
| 37 | `masses_plus_final_cls_diagnostic` | `diagnostic` | `rbf_svm_grid_01` | 0.999982 | 0.999996 | diagnostic only |
| 38 | `masses_plus_final_cls_diagnostic` | `diagnostic` | `extra_trees_03` | 0.999977 | 0.999995 | diagnostic only |
| 39 | `masses_plus_final_cls_diagnostic` | `diagnostic` | `rbf_svm_grid_02` | 0.999942 | 0.999987 | diagnostic only |
| 40 | `masses_plus_final_cls_diagnostic` | `diagnostic` | `logistic_polynomial_degree_3_03` | 0.999938 | 0.999986 | diagnostic only |
| 41 | `masses_plus_final_cls_diagnostic` | `diagnostic` | `logistic_polynomial_degree_3_02` | 0.999904 | 0.999978 | diagnostic only |
| 42 | `masses_plus_final_cls_diagnostic` | `diagnostic` | `knn_grid_02` | 0.999801 | 0.999955 | diagnostic only |
| 43 | `masses_plus_final_cls_diagnostic` | `diagnostic` | `knn_grid_04` | 0.999784 | 0.999951 | diagnostic only |
| 44 | `masses_plus_final_cls_diagnostic` | `diagnostic` | `knn_grid_01` | 0.999702 | 0.999920 | diagnostic only |
| 45 | `masses_plus_final_cls_diagnostic` | `diagnostic` | `logistic_polynomial_degree_3_01` | 0.999665 | 0.999925 | diagnostic only |
| 46 | `masses_plus_final_cls_diagnostic` | `diagnostic` | `knn_grid_03` | 0.999617 | 0.999902 | diagnostic only |
| 47 | `mass_engineered_v1` | `approved` | `hist_gradient_boosting_grid_01` | 0.975742 | 0.994361 | approved feature set |
| 48 | `mass_engineered_v1` | `approved` | `tuned_random_forest_04` | 0.975656 | 0.994422 | approved feature set |
| 49 | `mass_engineered_v1` | `approved` | `tuned_random_forest_03` | 0.975155 | 0.994315 | approved feature set |
| 50 | `mass_engineered_v1` | `approved` | `tuned_random_forest_02` | 0.974644 | 0.994155 | approved feature set |
| 51 | `mass_engineered_v1` | `approved` | `hist_gradient_boosting_grid_02` | 0.974071 | 0.993957 | approved feature set |
| 52 | `mass_engineered_v1` | `approved` | `hist_gradient_boosting_grid_03` | 0.973399 | 0.993781 | approved feature set |
| 53 | `mass_engineered_v1` | `approved` | `tuned_random_forest_01` | 0.972474 | 0.993585 | approved feature set |
| 54 | `mass_engineered_v1` | `approved` | `hist_gradient_boosting_grid_04` | 0.971898 | 0.993464 | approved feature set |
| 55 | `mass_engineered_v1` | `approved` | `extra_trees_02` | 0.971667 | 0.993487 | approved feature set |
| 56 | `mass_engineered_v1` | `approved` | `extra_trees_04` | 0.971525 | 0.993494 | approved feature set |
| 57 | `mass_engineered_v1` | `approved` | `extra_trees_03` | 0.968608 | 0.992765 | approved feature set |
| 58 | `mass_engineered_v1` | `approved` | `extra_trees_01` | 0.964836 | 0.991427 | approved feature set |
| 59 | `mass_engineered_v1` | `approved` | `knn_grid_04` | 0.961926 | 0.990242 | approved feature set |
| 60 | `mass_engineered_v1` | `approved` | `knn_grid_02` | 0.958845 | 0.988296 | approved feature set |
| 61 | `mass_engineered_v1` | `approved` | `knn_grid_03` | 0.958311 | 0.988920 | approved feature set |
| 62 | `mass_engineered_v1` | `approved` | `knn_grid_01` | 0.956968 | 0.986977 | approved feature set |
| 63 | `mass_engineered_v1` | `approved` | `logistic_polynomial_degree_3_03` | 0.953392 | 0.989495 | approved feature set |
| 64 | `mass_engineered_v1` | `approved` | `logistic_polynomial_degree_3_02` | 0.950396 | 0.988869 | approved feature set |
| 65 | `mass_engineered_v1` | `approved` | `logistic_polynomial_degree_3_01` | 0.947530 | 0.988267 | approved feature set |
| 66 | `mass_engineered_v1` | `approved` | `rbf_svm_grid_04` | 0.926401 | 0.983925 | approved feature set |
| 67 | `raw_masses` | `approved` | `knn_grid_04` | 0.925500 | 0.982186 | approved feature set |
| 68 | `raw_masses` | `approved` | `knn_grid_02` | 0.925217 | 0.980902 | approved feature set |
| 69 | `mass_engineered_v1` | `approved` | `rbf_svm_grid_02` | 0.923822 | 0.982934 | approved feature set |
| 70 | `mass_engineered_v1` | `approved` | `rbf_svm_grid_03` | 0.922823 | 0.983643 | approved feature set |
| 71 | `mass_engineered_v1` | `approved` | `rbf_svm_grid_01` | 0.921027 | 0.983150 | approved feature set |
| 72 | `raw_masses` | `approved` | `tuned_random_forest_02` | 0.919363 | 0.980956 | approved feature set |
| 73 | `raw_masses` | `approved` | `knn_grid_01` | 0.917804 | 0.977313 | approved feature set |
| 74 | `raw_masses` | `approved` | `extra_trees_02` | 0.916448 | 0.981240 | approved feature set |
| 75 | `raw_masses` | `approved` | `tuned_random_forest_04` | 0.915653 | 0.980106 | approved feature set |
| 76 | `raw_masses` | `approved` | `extra_trees_03` | 0.913621 | 0.980171 | approved feature set |
| 77 | `raw_masses` | `approved` | `extra_trees_04` | 0.911987 | 0.980275 | approved feature set |
| 78 | `raw_masses` | `approved` | `tuned_random_forest_01` | 0.911071 | 0.976985 | approved feature set |
| 79 | `raw_masses` | `approved` | `knn_grid_03` | 0.909094 | 0.977463 | approved feature set |
| 80 | `raw_masses` | `approved` | `tuned_random_forest_03` | 0.908655 | 0.978560 | approved feature set |
| 81 | `raw_masses` | `approved` | `extra_trees_01` | 0.906991 | 0.974200 | approved feature set |
| 82 | `raw_masses` | `approved` | `hist_gradient_boosting_grid_02` | 0.905349 | 0.978770 | approved feature set |
| 83 | `raw_masses` | `approved` | `hist_gradient_boosting_grid_04` | 0.904312 | 0.978551 | approved feature set |
| 84 | `raw_masses` | `approved` | `hist_gradient_boosting_grid_03` | 0.904128 | 0.978535 | approved feature set |
| 85 | `raw_masses` | `approved` | `hist_gradient_boosting_grid_01` | 0.903942 | 0.978419 | approved feature set |
| 86 | `raw_masses` | `approved` | `logistic_polynomial_degree_3_03` | 0.888954 | 0.974716 | approved feature set |
| 87 | `raw_masses` | `approved` | `logistic_polynomial_degree_3_02` | 0.888565 | 0.974627 | approved feature set |
| 88 | `raw_masses` | `approved` | `logistic_polynomial_degree_3_01` | 0.888013 | 0.974508 | approved feature set |
| 89 | `raw_masses` | `approved` | `rbf_svm_grid_02` | 0.885528 | 0.974085 | approved feature set |
| 90 | `raw_masses` | `approved` | `rbf_svm_grid_03` | 0.870140 | 0.971545 | approved feature set |
| 91 | `raw_masses` | `approved` | `rbf_svm_grid_01` | 0.851956 | 0.967633 | approved feature set |
| 92 | `raw_masses` | `approved` | `rbf_svm_grid_04` | 0.850109 | 0.966879 | approved feature set |

## Review Notes

- Diagnostic Final_CLs feature sets are possible leakage checks and are not thesis-accepted feature sets.
- ROC/AUC values use continuous scores only.
- All recommendations are provisional, unverified, pending review.


## AUC >= 0.97 Candidate Reproduction

The current candidate reproduction reports fixed seed-42 ROC-AUC = 0.975742 for `mass_engineered_v1` with histogram gradient boosting. This result is an observed model output and remains provisional, unverified, and pending review.

The candidate does not use `Final_CLs` as an approved feature. The reproduction script computes AUC from continuous `predict_proba` scores and applies balanced sample weighting to the training split only.

**Artifact note:** the existing generated candidate CSV and metadata were produced before the latest source fix that added full row-level status and explicit imbalance-strategy metadata. This notebook displays all result rows with the full provisional status, but the generated candidate artifacts should be regenerated after review if updated metadata files are required.

In [7]:
if candidate_metrics.empty:
    display(Markdown("**Missing candidate metrics CSV.** Candidate metrics remain pending generated evidence."))
else:
    metrics_display = candidate_metrics.copy()
    metrics_display["review_status_display"] = STATUS
    display(metrics_display[[
        "validation_type",
        "validation_id",
        "seed",
        "feature_set",
        "model_variant",
        "score_source",
        "roc_auc",
        "review_status_display",
    ]])

,validation_type,validation_id,seed,feature_set,model_variant,score_source,roc_auc,review_status_display
0,candidate_seed_42,seed_42,42,mass_engineered_v1,hist_gradient_boosting_grid_01,predict_proba_positive_class,0.975742,"provisional, unverified, pending review"
1,repeated_split,seed_11,11,mass_engineered_v1,hist_gradient_boosting_grid_01,predict_proba_positive_class,0.970412,"provisional, unverified, pending review"
2,repeated_split,seed_22,22,mass_engineered_v1,hist_gradient_boosting_grid_01,predict_proba_positive_class,0.964378,"provisional, unverified, pending review"
3,repeated_split,seed_33,33,mass_engineered_v1,hist_gradient_boosting_grid_01,predict_proba_positive_class,0.969928,"provisional, unverified, pending review"
4,repeated_split,seed_42,42,mass_engineered_v1,hist_gradient_boosting_grid_01,predict_proba_positive_class,0.975742,"provisional, unverified, pending review"
5,repeated_split,seed_55,55,mass_engineered_v1,hist_gradient_boosting_grid_01,predict_proba_positive_class,0.966158,"provisional, unverified, pending review"
6,repeated_split,seed_66,66,mass_engineered_v1,hist_gradient_boosting_grid_01,predict_proba_positive_class,0.973350,"provisional, unverified, pending review"
7,repeated_split,seed_77,77,mass_engineered_v1,hist_gradient_boosting_grid_01,predict_proba_positive_class,0.967716,"provisional, unverified, pending review"
8,repeated_split,seed_88,88,mass_engineered_v1,hist_gradient_boosting_grid_01,predict_proba_positive_class,0.968250,"provisional, unverified, pending review"
9,repeated_split,seed_99,99,mass_engineered_v1,hist_gradient_boosting_grid_01,predict_proba_positive_class,0.971005,"provisional, unverified, pending review"


## Robustness Status

Robustness across all repeated splits and cross-validation folds is not yet established. The candidate summary reports:

- repeated-split mean/std/min/max: 0.969264 / 0.003364 / 0.964378 / 0.975742
- cross-validation mean/std/min/max: 0.967593 / 0.000707 / 0.966636 / 0.968677
- AUC >= 0.97 robust across all repeated splits: False
- AUC >= 0.97 robust across CV folds: False

These values remain provisional, unverified, and pending review.

In [8]:
display_markdown_file(
    "outputs/runs/masses_exclusions_robustness_v1/robustness_summary.md",
    "Robustness Summary",
)

# Robustness Validation Summary

Status: provisional, unverified, pending review.

This summary reports observed validation outputs only. It is not an accepted thesis result.

| Validation | Feature set | Model variant | Mean AUC | Std | Min | Max | N | All > 0.97 | Mean > 0.97 |
| --- | --- | --- | ---: | ---: | ---: | ---: | ---: | --- | --- |
| `repeated_split` | `mass_engineered_v1` | `hist_gradient_boosting_grid_01_equivalent` | 0.969264 | 0.003364 | 0.964378 | 0.975742 | 10 | False | False |
| `cross_validation` | `mass_engineered_v1` | `hist_gradient_boosting_grid_01_equivalent` | 0.967593 | 0.000707 | 0.966636 | 0.968677 | 5 | False | False |
| `repeated_split` | `mass_engineered_v1` | `extra_trees_best_small_grid_01` | 0.966760 | 0.003737 | 0.961305 | 0.971667 | 10 | False | False |
| `repeated_split` | `mass_engineered_v1` | `extra_trees_best_small_grid_02` | 0.966455 | 0.003734 | 0.961392 | 0.971525 | 10 | False | False |
| `cross_validation` | `mass_engineered_v1` | `extra_trees_best_small_grid_01` | 0.964433 | 0.001396 | 0.962458 | 0.966663 | 5 | False | False |
| `cross_validation` | `mass_engineered_v1` | `extra_trees_best_small_grid_02` | 0.964216 | 0.001146 | 0.962586 | 0.965712 | 5 | False | False |
| `repeated_split` | `mass_engineered_v1` | `knn_grid_04_equivalent` | 0.954853 | 0.004781 | 0.946390 | 0.961926 | 10 | False | False |
| `cross_validation` | `mass_engineered_v1` | `knn_grid_04_equivalent` | 0.952941 | 0.001382 | 0.950914 | 0.954320 | 5 | False | False |
| `repeated_split` | `raw_masses` | `knn_grid_04_equivalent` | 0.924794 | 0.002742 | 0.919354 | 0.929465 | 10 | False | False |
| `cross_validation` | `raw_masses` | `knn_grid_04_equivalent` | 0.920034 | 0.006649 | 0.907157 | 0.925025 | 5 | False | False |
| `repeated_split` | `raw_masses` | `extra_trees_best_small_grid_01` | 0.914396 | 0.004158 | 0.907145 | 0.920981 | 10 | False | False |
| `cross_validation` | `raw_masses` | `extra_trees_best_small_grid_01` | 0.911114 | 0.006483 | 0.900529 | 0.920122 | 5 | False | False |
| `repeated_split` | `raw_masses` | `extra_trees_best_small_grid_02` | 0.909764 | 0.004499 | 0.902087 | 0.917350 | 10 | False | False |
| `cross_validation` | `raw_masses` | `extra_trees_best_small_grid_02` | 0.906881 | 0.006754 | 0.895506 | 0.915740 | 5 | False | False |
| `repeated_split` | `raw_masses` | `hist_gradient_boosting_grid_01_equivalent` | 0.904547 | 0.004325 | 0.894997 | 0.912097 | 10 | False | False |
| `cross_validation` | `raw_masses` | `hist_gradient_boosting_grid_01_equivalent` | 0.902137 | 0.006141 | 0.890107 | 0.907164 | 5 | False | False |

All recommendations are provisional, unverified, pending review.


## Diagnostic Final_CLs Warning

`Final_CLs` is treated as an audit-only column in this workflow. It may be useful for diagnostic checks, including possible leakage or label-construction behavior, but it is not an approved feature or target for the candidate model.

Diagnostic `Final_CLs` AUC values must not be read as accepted thesis evidence. Promoting `Final_CLs` to an approved feature or target would require explicit scientific review.

## Interpretation

The present evidence supports a promising engineered-mass machine-learning candidate on the fixed seed-42 split. It does not yet establish robust AUC >= 0.97 across repeated splits or cross-validation, and it does not establish a final symbolic PySR result.

No physics conclusion, dataset convention, symbolic expression, or final thesis claim is accepted from this notebook alone. Those decisions remain subject to human and supervisor review.

## AI-Assisted Research Workflow Note

Codex-CLI was used as a repo-side AI-assisted research workflow assistant to accelerate reproducible development. Its role included script scaffolding, configuration organization, verification checks, Git-tracked checkpoints, and implementation review within the repository workflow.

Codex was not treated as a scientific authority. All model results, physics interpretations, dataset conventions, and supervisor-facing claims remain provisional until reviewed and accepted by the thesis author and supervisor.

## Next Steps

1. Review unresolved dataset conventions, including mass units, binary label semantics, preprocessing rules, and acceptable validation protocol.
2. Review whether the engineered mass features in `mass_engineered_v1` are scientifically acceptable for supervisor-facing comparison.
3. Regenerate the AUC candidate outputs after the latest script/config fixes if updated metadata files are required.
4. Review robustness results before launching the final PySR symbolic-search step.
5. If approved, run PySR under a reviewed config and compare its continuous-score ROC/AUC against the standard ML baselines.
6. Prepare a formal review record before treating any result as accepted.

## Appendix: Reproducibility Commands

These commands document the current reproducibility workflow. They are not executed by this notebook.

```bash
python scripts/audit_dataset.py --config configs/runs/masses_exclusions_audit.yaml
python scripts/evaluate_strong_baselines.py --config configs/runs/masses_exclusions_strong_baselines.yaml
python scripts/evaluate_robustness.py --config configs/runs/masses_exclusions_robustness.yaml
python scripts/reproduce_auc97_candidate.py --config configs/runs/masses_exclusions_auc97_candidate.yaml
python scripts/train_pysr_auc_search.py --config configs/runs/masses_exclusions_pysr_search.yaml --dry-run
```

The PySR search should remain blocked until the relevant dataset conventions, feature conventions, validation protocol, and robustness status are reviewed.

In [9]:
evidence_paths = [
    "configs/datasets/masses_exclusions.yaml",
    "configs/runs/masses_exclusions_auc97_candidate.yaml",
    "outputs/runs/masses_exclusions_audit/dataset_audit.md",
    "outputs/runs/masses_exclusions_strong_baselines_v1/strong_summary.md",
    "outputs/runs/masses_exclusions_robustness_v1/robustness_summary.md",
    "outputs/runs/masses_exclusions_auc97_candidate/auc97_candidate_summary.md",
    "outputs/runs/masses_exclusions_auc97_candidate/auc97_candidate_metrics.csv",
    "outputs/runs/masses_exclusions_auc97_candidate/auc97_candidate_metadata.json",
]
display(pd.DataFrame([
    {"path": path, "exists": repo_path(path).exists()}
    for path in evidence_paths
]))

metadata_rows = []
if candidate_metadata is not None:
    metadata_rows = [
        {"field": "run_id", "value": candidate_metadata.get("run_id")},
        {"field": "dataset_id", "value": candidate_metadata.get("dataset_id")},
        {"field": "command", "value": candidate_metadata.get("command")},
        {"field": "code_version_or_commit", "value": candidate_metadata.get("code_version_or_commit")},
        {"field": "status", "value": candidate_metadata.get("status")},
        {"field": "class_imbalance_strategy", "value": candidate_metadata.get("class_imbalance_strategy")},
    ]
display(pd.DataFrame(metadata_rows))

if candidate_metadata is not None and candidate_metadata.get("class_imbalance_strategy") is None:
    display(Markdown(
        "**Metadata warning:** the existing generated candidate metadata lacks "
        "`class_imbalance_strategy`, indicating it predates the latest source/config fix. "
        f"Treat the generated artifact as {STATUS}."
    ))

,path,exists
0,configs/datasets/masses_exclusions.yaml,True
1,configs/runs/masses_exclusions_auc97_candidate...,True
2,outputs/runs/masses_exclusions_audit/dataset_a...,True
3,outputs/runs/masses_exclusions_strong_baseline...,True
4,outputs/runs/masses_exclusions_robustness_v1/r...,True
5,outputs/runs/masses_exclusions_auc97_candidate...,True
6,outputs/runs/masses_exclusions_auc97_candidate...,True
7,outputs/runs/masses_exclusions_auc97_candidate...,True


,field,value
0,run_id,masses_exclusions_auc97_candidate
1,dataset_id,masses_exclusions
2,command,scripts/reproduce_auc97_candidate.py --config ...
3,code_version_or_commit,6b4af4d8f41b840bd734649b8f511a35c2158f21
4,status,"provisional, unverified, pending review"
5,class_imbalance_strategy,NaN


**Metadata warning:** the existing generated candidate metadata lacks `class_imbalance_strategy`, indicating it predates the latest source/config fix. Treat the generated artifact as provisional, unverified, pending review.